# Task 6: Build the merged v0 dataset

We want one row per job, with its exposure scores and O*NET information together.

- **Join key:** six-digit 2018 SOC, stored as text, for example `15-1252`.
- **Column name:** `soc2018` in every table.
- **Base table:** `eloundou_6digit.csv`, the 798-job version of `occ_level.csv` made in Task 2.
- **Main join:** left join, keeping every job in the base table.
- **Coverage check:** full outer join of the job-code lists, showing jobs present in any source.

The earlier preparation notebooks document the 2018 SOC alignment. The checks here verify code format, missing codes, uniqueness, and overlap; code format alone does not establish the SOC year.

This notebook logs each join, resolves repeated job codes, checks five selected occupations against source data, and exports `merged_v0.csv` and `data_dictionary.csv` to `data/datasets/processed/`.

**Selected v0 methods:** retain all 798 Eloundou jobs using left joins, check jobs outside that base separately, and use the simple unweighted mean when multiple source scores map to one 2018 job code.

## Hurdle carried over from Task 4

The crosswalk connected old job codes to newer ones. Sometimes several old jobs mapped to the same new job, leaving multiple scores for one 2018 code.

- AIOE has **24 codes with multiple rows**.
- Frey–Osborne has **13 codes with multiple rows**.
- AIOE also has **one row without a 2018 code**: Biologists (`19-1020`).

Joining these files directly would duplicate jobs. We first inspect the repeated codes, then make one row per code. The unmatched AIOE row is kept separately so it remains accounted for.

In [1]:
import pandas as pd
from pathlib import Path

data_folder = Path('../data/datasets')
if not data_folder.is_dir():
    data_folder = Path('data/datasets')

processed_folder = data_folder / 'processed'


In [2]:
# Load job codes as text to preserve their format
eloundou = pd.read_csv(data_folder / 'eloundou_6digit.csv', dtype={'soc_6digit': 'string'})
aioe = pd.read_csv(processed_folder / 'aioe_2018soc.csv', dtype={'soc2010': 'string', 'soc2018': 'string'})
frey = pd.read_csv(processed_folder / 'frey_osborne_2018soc.csv', dtype={'soc2010': 'string', 'soc2018': 'string'})

onet_files = {
    'job_zones': 'onet_job_zones_6digit.csv',
    'abilities': 'onet_abilities_6digit.csv',
    'skills': 'onet_skills_6digit.csv',
    'work_activities': 'onet_work_activities_6digit.csv',
    'work_context': 'onet_work_context_6digit.csv',
}

onet_tables = {}
for name, filename in onet_files.items():
    onet_tables[name] = pd.read_csv(processed_folder / filename, dtype={'soc_6digit': 'string'})

eloundou.head()

,soc_6digit,Title,dv_rating_alpha,dv_rating_beta,dv_rating_gamma,human_rating_alpha,human_rating_beta,human_rating_gamma
0,11-1011,Chief Executives,0.133333,0.507778,0.882222,0.117778,0.369444,0.621111
1,11-1021,General and Operations Managers,0.000000,0.480769,0.961538,0.115385,0.384615,0.653846
2,11-1031,Legislators,0.033333,0.400000,0.766667,0.266667,0.516667,0.766667
3,11-2011,Advertising and Promotions Managers,0.000000,0.476744,0.953488,0.255814,0.546512,0.837209
4,11-2021,Marketing Managers,0.062500,0.500000,0.937500,0.218750,0.578125,0.937500


In [3]:
list(onet_tables)

['job_zones', 'abilities', 'skills', 'work_activities', 'work_context']

In [4]:
# Use the same job-code column name in every table
eloundou = eloundou.rename(columns={'soc_6digit': 'soc2018'})
for name in onet_tables:
    onet_tables[name] = onet_tables[name].rename(columns={'soc_6digit': 'soc2018'})

tables = {'eloundou': eloundou, 'aioe': aioe, 'frey': frey, **onet_tables}

key_checks = []
for name, table in tables.items():
    table['soc2018'] = table['soc2018'].str.strip().replace('', pd.NA)
    codes = table['soc2018'].dropna()
    title_column = next((column for column in ['Title', 'title_2018', 'title'] if column in table), None)
    invalid_codes = ~codes.str.fullmatch(r'[0-9]{2}-[0-9]{4}')
    key_checks.append({
        'source': name,
        'rows': len(table),
        'unique_codes': codes.nunique(),
        'missing_codes': int(table['soc2018'].isna().sum()),
        'missing_titles': int(table[title_column].fillna('').str.strip().eq('').sum()) if title_column else pd.NA,
        'codes_with_multiple_rows': int((codes.value_counts() > 1).sum()),
        'invalid_format': int(invalid_codes.sum()),
    })
    assert not invalid_codes.any(), f'Check the job-code format in {name}.'

key_summary = pd.DataFrame(key_checks)
key_summary

,source,rows,unique_codes,missing_codes,missing_titles,codes_with_multiple_rows,invalid_format
0,eloundou,798,798,0,24,0,0
1,aioe,827,800,1,1,24,0
2,frey,681,668,0,0,13,0
3,job_zones,798,798,0,0,0,0
4,abilities,774,774,0,0,0,0
5,skills,774,774,0,0,0,0
6,work_activities,774,774,0,0,0,0
7,work_context,774,774,0,0,0,0


## 1. Inspect the repeated and unmatched codes

These tables keep the original source rows, so we can see which old jobs contributed to each new job.

In [5]:
aioe_duplicates = aioe[aioe['soc2018'].notna() & aioe['soc2018'].duplicated(keep=False)].sort_values('soc2018')
frey_duplicates = frey[frey['soc2018'].notna() & frey['soc2018'].duplicated(keep=False)].sort_values('soc2018')

print('AIOE codes with multiple rows:', aioe_duplicates['soc2018'].nunique())
print('Frey–Osborne codes with multiple rows:', frey_duplicates['soc2018'].nunique())

aioe_duplicates[['soc2010', 'title_2010', 'soc2018', 'title_2018', 'aioe']].head(10)

AIOE codes with multiple rows: 24
Frey–Osborne codes with multiple rows: 13


,soc2010,title_2010,soc2018,title_2018,aioe
37,11-9199,"Managers, All Other",13-1082,Project Management Specialists,0.964687
56,13-1199,"Business Operations Specialists, All Other",13-1082,Project Management Specialists,1.047687
90,15-1199,"Computer Occupations, All Other",13-1082,Project Management Specialists,1.232705
64,13-2051,Financial Analysts,13-2054,Financial Risk Specialists,1.380615
72,13-2099,"Financial Specialists, All Other",13-2054,Financial Risk Specialists,1.361362
85,15-1141,Database Administrators,15-1243,Database Architects,1.284863
91,15-1199,"Computer Occupations, All Other",15-1243,Database Architects,1.232705
78,15-1132,"Software Developers, Applications",15-1252,Software Developers,1.200923
80,15-1133,"Software Developers, Systems Software",15-1252,Software Developers,1.283308
81,15-1133,"Software Developers, Systems Software",15-1253,Software Quality Assurance Analysts and Testers,1.283308


In [6]:
frey_duplicates[['soc2010', 'title_2010', 'soc2018', 'title_2018', 'fo_prob']].head(10)

,soc2010,title_2010,soc2018,title_2018,fo_prob
35,11-9199,"Managers, All Other",13-1082,Project Management Specialists,0.250
48,13-1199,"Business Operations Specialists, All Other",13-1082,Project Management Specialists,0.230
56,13-2051,Financial Analysts,13-2054,Financial Risk Specialists,0.230
64,13-2099,"Financial Specialists, All Other",13-2054,Financial Risk Specialists,0.330
69,15-1132,"Software Developers, Applications",15-1252,Software Developers,0.042
71,15-1133,"Software Developers, Systems Software",15-1252,Software Developers,0.130
70,15-1132,"Software Developers, Applications",15-1253,Software Quality Assurance Analysts and Testers,0.042
72,15-1133,"Software Developers, Systems Software",15-1253,Software Quality Assurance Analysts and Testers,0.130
153,19-4041,Geological and Petroleum Technicians,19-4044,Hydrologic Technicians,0.910
159,19-4099,"Life, Physical, and Social Science Technicians...",19-4044,Hydrologic Technicians,0.610


In [7]:
# Keep unmatched source rows separate; never give them a made-up code
aioe_unmatched = aioe[aioe['soc2018'].isna()].copy()
frey_unmatched = frey[frey['soc2018'].isna()].copy()

print('Unmatched AIOE rows:', len(aioe_unmatched))
print('Unmatched Frey–Osborne rows:', len(frey_unmatched))
aioe_unmatched[['soc2010', 'title_2010', 'aioe']]

Unmatched AIOE rows: 1
Unmatched Frey–Osborne rows: 0


,soc2010,title_2010,aioe
138,19-1020,Biologists,0.752081


## 2. Make one row per 2018 job code

**Selected v0 rule:** take the simple, unweighted mean when several source scores map to the same 2018 job. Scores are not counts, so we do not sum them. No employment weights are supplied for this step.

This is a practical assumption, not an exact reconstruction of the new occupation's score. It is especially approximate when an old job was both split and merged. The team can revisit the rule later.

Task 4 already copied a score to each child when an old job split; we retain that treatment. We keep the number of contributing source rows and a flag for any non-1:1 mapping. The original `aioe` and `frey` tables remain available for tracing those mappings.

In [8]:
aioe_matched = aioe[aioe['soc2018'].notna()].copy()
frey_matched = frey[frey['soc2018'].notna()].copy()

aioe_matched['crosswalk_changed'] = aioe_matched['match_type'].ne('1:1')
frey_matched['crosswalk_changed'] = frey_matched['match_type'].ne('1:1')

aioe_ready = aioe_matched.groupby('soc2018', as_index=False).agg(
    aioe=('aioe', 'mean'),
    aioe_source_rows=('soc2010', 'size'),
    aioe_source_codes=('soc2010', lambda codes: ';'.join(sorted(codes.unique()))),
    aioe_crosswalk_changed=('crosswalk_changed', 'any'),
)

frey_ready = frey_matched.groupby('soc2018', as_index=False).agg(
    fo_prob=('fo_prob', 'mean'),
    fo_source_rows=('soc2010', 'size'),
    fo_source_codes=('soc2010', lambda codes: ';'.join(sorted(codes.unique()))),
    fo_crosswalk_changed=('crosswalk_changed', 'any'),
)

print('AIOE: one row each for', len(aioe_ready), 'job codes')
print('Frey–Osborne: one row each for', len(frey_ready), 'job codes')
aioe_ready.head()

AIOE: one row each for 800 job codes
Frey–Osborne: one row each for 668 job codes


,soc2018,aioe,aioe_source_rows,aioe_source_codes,aioe_crosswalk_changed
0,11-1011,1.334246,1,11-1011,False
1,11-1021,0.574877,1,11-1021,False
2,11-2011,1.294387,1,11-2011,False
3,11-2021,1.315032,1,11-2021,False
4,11-2022,1.266280,1,11-2022,False


## 3. Join one dataset at a time

Keep all 798 Eloundou jobs and add matching scores and job characteristics. Each step reports the starting row count, ending row count, matched jobs, and unmatched jobs. `validate='one_to_one'` rejects repeated keys.

**Task 2 title gap:** the original BLS crosswalk supplies official broad 2018 titles for all 24 unnamed groups. The final `title` uses this official lookup for every job. `eloundou_title` preserves the original labels, including its 24 blanks. `onet_title` and `onet_title_rollup_method` retain O*NET labels and explain whether they came from a base entry or a specialty. Earlier source files are not changed.

In [9]:
master = eloundou.rename(columns={'Title': 'eloundou_title'}).copy()
assert master["soc2018"].notna().all()

# Read the official 2018 codes and titles from the original BLS workbook
crosswalk_raw = pd.read_excel(data_folder / 'soc_2010_to_2018_crosswalk.xlsx', header=8, dtype='string')
for column in ['2010 SOC Code', '2018 SOC Code']:
    crosswalk_raw[column] = crosswalk_raw[column].str.strip()

soc_titles = crosswalk_raw[['2018 SOC Code', '2018 SOC Title']].rename(
    columns={'2018 SOC Code': 'soc2018', '2018 SOC Title': 'title'}
).copy()
soc_titles['title'] = soc_titles['title'].str.replace(r'\s*\(#+\)', '', regex=True).str.strip()
soc_titles = soc_titles.dropna().drop_duplicates()

# Check actual membership in the BLS 2018 classification, not just code format
official_codes = set(soc_titles['soc2018'])
for name, table in tables.items():
    unknown_codes = set(table['soc2018'].dropna()) - official_codes
    assert not unknown_codes, f'{name}: codes absent from BLS 2018: {unknown_codes}'

join_tables = {'bls_titles': soc_titles, 'aioe': aioe_ready, 'frey': frey_ready}
for name, table in onet_tables.items():
    if name == 'job_zones':
        table = table.rename(columns={'title': 'onet_title', 'title_rollup_method': 'onet_title_rollup_method'})
    else:
        table = table.drop(columns=['title', 'title_rollup_method'])
    join_tables[name] = table

join_records = []
for name, table in join_tables.items():
    assert table["soc2018"].notna().all()
    rows_before = len(master)
    master = master.merge(table, on='soc2018', how='left', validate='one_to_one', indicator='_match')
    matched_jobs = int(master['_match'].eq('both').sum())
    join_records.append({
        'source': name,
        'rows_before': rows_before,
        'rows_after': len(master),
        'matched_jobs': matched_jobs,
        'unmatched_jobs': int(master['_match'].eq('left_only').sum()),
    })
    print(f'{name}: {rows_before} -> {len(master)} rows; {matched_jobs} matched; {len(master) - matched_jobs} unmatched')
    master = master.drop(columns='_match')
    assert len(master) == rows_before, f'{name} changed the number of jobs.'

assert set(master['soc2018']) == set(eloundou['soc2018'])

# Preserve missing information rather than turning it into zero or False
for column in ['aioe_source_rows', 'fo_source_rows']:
    master[column] = master[column].astype('Int64')
for column in ['aioe_crosswalk_changed', 'fo_crosswalk_changed']:
    master[column] = master[column].astype('boolean')

join_log = pd.DataFrame(join_records)
join_log

bls_titles: 798 -> 798 rows; 798 matched; 0 unmatched
aioe: 798 -> 798 rows; 790 matched; 8 unmatched
frey: 798 -> 798 rows; 663 matched; 135 unmatched
job_zones: 798 -> 798 rows; 798 matched; 0 unmatched
abilities: 798 -> 798 rows; 774 matched; 24 unmatched
skills: 798 -> 798 rows; 774 matched; 24 unmatched
work_activities: 798 -> 798 rows; 774 matched; 24 unmatched
work_context: 798 -> 798 rows; 774 matched; 24 unmatched


,source,rows_before,rows_after,matched_jobs,unmatched_jobs
0,bls_titles,798,798,798,0
1,aioe,798,798,790,8
2,frey,798,798,663,135
3,job_zones,798,798,798,0
4,abilities,798,798,774,24
5,skills,798,798,774,24
6,work_activities,798,798,774,24
7,work_context,798,798,774,24


## 4. Full outer join for the coverage check

Join only the unique code lists here. Each flag says whether a job is present in a source, not whether every score is filled in. This includes jobs outside the Eloundou base. Rows without a code stay in the separate unmatched tables above.

In [10]:
ready_tables = {'eloundou': eloundou, 'aioe': aioe_ready, 'frey': frey_ready, **onet_tables}
coverage = eloundou[['soc2018']].copy()
coverage['in_eloundou'] = True

for name, table in ready_tables.items():
    if name == 'eloundou':
        continue
    source_codes = table[['soc2018']].copy()
    source_codes['in_' + name] = True
    coverage = coverage.merge(source_codes, on='soc2018', how='outer', validate='one_to_one')

presence_columns = [column for column in coverage.columns if column.startswith('in_')]
coverage[presence_columns] = coverage[presence_columns].eq(True)
coverage = coverage.sort_values('soc2018').reset_index(drop=True)

outside_base = coverage[~coverage['in_eloundou']].copy()

print('Distinct job codes across all sources:', len(coverage))
print('Jobs outside the Eloundou base:', len(outside_base))
outside_base

Distinct job codes across all sources: 808
Jobs outside the Eloundou base: 10


,soc2018,in_eloundou,in_aioe,in_frey,in_job_zones,in_abilities,in_skills,in_work_activities,in_work_context
23,11-9039,False,True,False,False,False,False,False,False
249,25-3099,False,True,False,False,False,False,False,False
260,25-9049,False,True,True,False,False,False,False,False
290,27-3099,False,True,True,False,False,False,False,False
338,29-1249,False,True,False,False,False,False,False,False
487,43-2099,False,True,True,False,False,False,False,False
602,47-5049,False,True,True,False,False,False,False,False
757,51-9199,False,True,False,False,False,False,False,False
762,53-1049,False,True,False,False,False,False,False,False
807,53-7199,False,True,True,False,False,False,False,False


In [11]:
# Source coverage among the jobs retained in the main table
base_coverage = coverage[coverage['in_eloundou']]
coverage_summary = pd.DataFrame({
    'matched_base_jobs': base_coverage[presence_columns].sum(),
    'missing_base_jobs': (~base_coverage[presence_columns]).sum(),
})
coverage_summary.index.name = 'source'
coverage_summary

,matched_base_jobs,missing_base_jobs
source,,
in_eloundou,798,0
in_aioe,790,8
in_frey,663,135
in_job_zones,798,0
in_abilities,774,24
in_skills,774,24
in_work_activities,774,24
in_work_context,774,24


In [12]:
# Verify the 24 formerly unnamed groups now have official broad titles
title_repairs = master.loc[master['eloundou_title'].isna(), ['soc2018', 'eloundou_title', 'title', 'onet_title']]
print('Original missing Eloundou titles:', len(title_repairs))
print('Missing final titles:', int(master['title'].isna().sum()))
title_repairs

Original missing Eloundou titles: 24
Missing final titles: 0


,soc2018,eloundou_title,title,onet_title
35,11-9179,NaN,"Personal Service Managers, All Other",Fitness and Wellness Coordinators
36,11-9199,NaN,"Managers, All Other",Regulatory Affairs Managers
56,13-1199,NaN,"Business Operations Specialists, All Other",Business Continuity Planners
71,13-2099,NaN,"Financial Specialists, All Other",Financial Quantitative Analysts
86,15-1299,NaN,"Computer Occupations, All Other",Web Administrators
92,15-2099,NaN,"Mathematical Science Occupations, All Other",Bioinformatics Technicians
114,17-2199,NaN,"Engineers, All Other","Energy Engineers, Except Wind and Solar"
126,17-3029,NaN,"Engineering Technologists and Technicians, Exc...",Non-Destructive Testing Specialists
134,19-1029,NaN,"Biological Scientists, All Other",Bioinformatics Scientists
147,19-2099,NaN,"Physical Scientists, All Other",Remote Sensing Scientists and Technologists


In [13]:
# Add source-presence flags to the deliverable
master = master.merge(coverage, on='soc2018', how='left', validate='one_to_one')
first_columns = ['soc2018', 'title', 'eloundou_title', 'onet_title']
master = master[first_columns + [column for column in master.columns if column not in first_columns]]

print('Final shape:', master.shape)
master[['soc2018', 'title', 'dv_rating_beta', 'human_rating_beta', 'aioe', 'fo_prob', 'job_zone']].head()

Final shape: (798, 214)


,soc2018,title,dv_rating_beta,human_rating_beta,aioe,fo_prob,job_zone
0,11-1011,Chief Executives,0.507778,0.369444,1.334246,0.015,5
1,11-1021,General and Operations Managers,0.480769,0.384615,0.574877,0.160,4
2,11-1031,Legislators,0.400000,0.516667,NaN,NaN,4
3,11-2011,Advertising and Promotions Managers,0.476744,0.546512,1.294387,0.039,4
4,11-2021,Marketing Managers,0.500000,0.578125,1.315032,0.014,4


## 5. Build the data dictionary

Every final column gets a source, plain-language description, data type, and missing-value count. Feature descriptions reuse the Task 5 dictionary so each O*NET element and scale remains traceable.

AIOE is a standardized exposure score, Frey–Osborne is a computerisation probability, and Eloundou measures task exposure. Their raw scores are not interchangeable. This v0 preserves the scores; ranking and analysis come later.

In [14]:
feature_dictionary = pd.read_csv(processed_folder / 'onet_feature_dictionary.csv')
assert feature_dictionary['column_name'].is_unique

# Metadata and scores that are not O*NET feature columns
column_notes = {
    'soc2018': ('eloundou_6digit.csv; BLS crosswalk', 'Six-digit 2018 SOC job code stored as text; one row per code.'),
    'title': ('soc_2010_to_2018_crosswalk.xlsx', 'Official broad 2018 SOC title; BLS footnote markers removed.'),
    'eloundou_title': ('eloundou_6digit.csv', 'Original Task 2 title from the .00 base entry; blank for 24 groups without a base entry.'),
    'onet_title': ('onet_job_zones_6digit.csv', 'O*NET title from the .00 entry, or the first detailed occupation when no base exists; may be narrower than the broad SOC title.'),
    'onet_title_rollup_method': ('onet_job_zones_6digit.csv', 'base_00 or first_detail; identifies how the O*NET title was selected.'),
    'aioe': ('aioe_2018soc.csv; AIOE_appendixA.xlsx', 'Standardized AI Occupational Exposure score; unweighted mean of contributing 2010 scores for this 2018 code. Not a probability.'),
    'aioe_source_rows': ('aioe_2018soc.csv', 'Number of matched source rows averaged for AIOE; blank when unmatched.'),
    'aioe_source_codes': ('aioe_2018soc.csv', 'Semicolon-separated contributing 2010 SOC codes; blank when unmatched.'),
    'aioe_crosswalk_changed': ('aioe_2018soc.csv', 'True if any contributing mapping is not 1:1; False for direct mappings; blank when unmatched.'),
    'fo_prob': ('frey_osborne_2018soc.csv; frey_osborne_probabilities.csv', 'Computerisation probability, 0–1; unweighted mean of contributing 2010 scores. The supplied source covers 653 occupations, not the full 702-paper list.'),
    'fo_source_rows': ('frey_osborne_2018soc.csv', 'Number of matched source rows averaged for Frey–Osborne; blank when unmatched.'),
    'fo_source_codes': ('frey_osborne_2018soc.csv', 'Semicolon-separated contributing 2010 SOC codes; blank when unmatched.'),
    'fo_crosswalk_changed': ('frey_osborne_2018soc.csv', 'True if any contributing mapping is not 1:1; False for direct mappings; blank when unmatched.'),
    'job_zone': ('onet_job_zones_6digit.csv; onet_job_zones.xlsx', 'Preparation category 1–5. Use the .00 entry when present; otherwise the most common detailed value, choosing the lower value on a tie.'),
    'job_zone_rollup_method': ('onet_job_zones_6digit.csv', 'base_00, detail_mode, or detail_mode_tie_lower; describes the Job Zone selection.'),
    'job_zone_had_conflict': ('onet_job_zones_6digit.csv', 'True when detailed occupations within this six-digit group have different Job Zones.'),
    'job_zone_source_value_count': ('onet_job_zones_6digit.csv', 'Number of distinct Job Zone values in the source group; not a count of occupations.'),
}

exposure_definitions = {
    'alpha': 'E1 share of tasks',
    'beta': 'E1 + 0.5 × E2 share of tasks',
    'gamma': 'E1 + E2 share of tasks; called zeta in the paper',
}
for rater, label in [('dv', 'GPT-4'), ('human', 'Human annotator')]:
    for measure, meaning in exposure_definitions.items():
        column_notes[f'{rater}_rating_{measure}'] = (
            'occ_level.csv via eloundou_6digit.csv',
            f'{label} exposure: {meaning}, range 0–1; simple mean across detailed occupations within the six-digit code.',
        )

for name in ready_tables:
    source = 'eloundou_6digit.csv' if name == 'eloundou' else (
        'aioe_2018soc.csv' if name == 'aioe' else (
            'frey_osborne_2018soc.csv' if name == 'frey' else onet_files[name]
        )
    )
    column_notes['in_' + name] = (source, 'True if this job code is present in the source; False if absent. Presence does not guarantee every value is populated.')

for feature in feature_dictionary.itertuples(index=False):
    scale = 'Importance (IM)' if feature.scale_id == 'IM' else 'continuous Context (CX)'
    column_notes[feature.column_name] = (
        feature.source_workbook,
        f'O*NET {feature.element_name}; element {feature.element_id}; {scale}. {feature.rollup_rule}. Blank when the source has no rating for this occupation.',
    )

assert set(master.columns) == set(column_notes), 'Dictionary and final columns do not match.'
dictionary_rows = []
for column in master.columns:
    source, description = column_notes[column]
    dictionary_rows.append({
        'column_name': column,
        'source': source,
        'description': description,
        'data_type': str(master[column].dtype),
        'missing_values': int(master[column].isna().sum()),
    })

data_dictionary = pd.DataFrame(dictionary_rows)
print('Documented columns:', len(data_dictionary))
data_dictionary.head(12)

Documented columns: 214


,column_name,source,description,data_type,missing_values
0,soc2018,eloundou_6digit.csv; BLS crosswalk,Six-digit 2018 SOC job code stored as text; on...,string,0
1,title,soc_2010_to_2018_crosswalk.xlsx,Official broad 2018 SOC title; BLS footnote ma...,string,0
2,eloundou_title,eloundou_6digit.csv,Original Task 2 title from the .00 base entry;...,object,24
3,onet_title,onet_job_zones_6digit.csv,"O*NET title from the .00 entry, or the first d...",object,0
4,dv_rating_alpha,occ_level.csv via eloundou_6digit.csv,"GPT-4 exposure: E1 share of tasks, range 0–1; ...",float64,0
5,dv_rating_beta,occ_level.csv via eloundou_6digit.csv,"GPT-4 exposure: E1 + 0.5 × E2 share of tasks, ...",float64,0
6,dv_rating_gamma,occ_level.csv via eloundou_6digit.csv,GPT-4 exposure: E1 + E2 share of tasks; called...,float64,0
7,human_rating_alpha,occ_level.csv via eloundou_6digit.csv,"Human annotator exposure: E1 share of tasks, r...",float64,0
8,human_rating_beta,occ_level.csv via eloundou_6digit.csv,Human annotator exposure: E1 + 0.5 × E2 share ...,float64,0
9,human_rating_gamma,occ_level.csv via eloundou_6digit.csv,Human annotator exposure: E1 + E2 share of tas...,float64,0


## 6. Internal QA: five deliberately selected occupations

These examples cover different join situations:

1. **11-1011 — Chief Executives:** direct crosswalk match, with multiple Eloundou detail rows to average.
2. **11-9179 — Personal Service Managers, All Other:** no .00 base, two specialties, and a copied score from a split.
3. **13-1082 — Project Management Specialists:** several old occupations contribute scores to one new occupation.
4. **11-2032 — Public Relations Managers:** a new occupation created by splitting an old code.
5. **29-1299 — Healthcare Diagnosing or Treating Practitioners, All Other:** no .00 base and absent O*NET feature ratings, which must stay missing.

For each example, reload the original source files. Recalculate all six Eloundou scores, both crosswalked indices, Job Zone, and all 183 O*NET features, then compare them with the final row. Also verify the official title and contributing source-code lists. A missing source value must remain missing; it is not replaced with zero.

In [15]:
qa_codes = ['11-1011', '11-9179', '13-1082', '11-2032', '29-1299']

raw_eloundou = pd.read_csv(data_folder / 'occ_level.csv', dtype={'O*NET-SOC Code': 'string'})
raw_eloundou['soc2018'] = raw_eloundou['O*NET-SOC Code'].str.split('.').str[0]
raw_aioe = pd.read_excel(data_folder / 'AIOE_appendixA.xlsx', dtype={'SOC Code': 'string'})
raw_frey = pd.read_csv(data_folder / 'frey_osborne_probabilities.csv', dtype={'soc_code_2010': 'string'})
raw_zones = pd.read_excel(data_folder / 'onet_job_zones.xlsx')
raw_zones['soc2018'] = raw_zones['O*NET-SOC Code'].str.split('.').str[0]

# Keep just the five selected occupations after reading each raw feature file
raw_features = {}
for filename in feature_dictionary['source_workbook'].unique():
    source = pd.read_excel(data_folder / filename, usecols=['O*NET-SOC Code', 'Element ID', 'Scale ID', 'Data Value'])
    source['soc2018'] = source['O*NET-SOC Code'].str.split('.').str[0]
    raw_features[filename] = source[source['soc2018'].isin(qa_codes)].copy()

print('Loaded original sources for the five QA occupations.')

Loaded original sources for the five QA occupations.


In [16]:
import math

qa_records = []

# One small helper keeps the comparisons readable
def check_value(soc, column, expected, source_file, source_rows):
    actual = master.loc[master['soc2018'].eq(soc), column].iloc[0]
    if pd.isna(expected):
        passed = bool(pd.isna(actual))
    elif pd.isna(actual):
        passed = False
    elif isinstance(expected, str):
        passed = actual == expected
    else:
        passed = math.isclose(float(actual), float(expected), rel_tol=1e-9, abs_tol=1e-10)
    qa_records.append({
        'soc2018': soc, 'column_name': column, 'source_file': source_file,
        'source_rows': source_rows, 'expected': expected, 'actual': actual, 'passed': passed,
    })

rating_columns = [column for column in raw_eloundou if '_rating_' in column]
for soc in qa_codes:
    details = raw_eloundou[raw_eloundou['soc2018'].eq(soc)]
    detail_codes = ';'.join(details['O*NET-SOC Code'])
    for column in rating_columns:
        check_value(soc, column, details[column].mean(), 'occ_level.csv', detail_codes)

    # Independently trace the BLS mapping back to the original 2010 scores
    mappings = crosswalk_raw[crosswalk_raw['2018 SOC Code'].eq(soc)]
    old_codes = mappings['2010 SOC Code'].unique()
    aioe_sources = raw_aioe[raw_aioe['SOC Code'].isin(old_codes)]
    frey_sources = raw_frey[raw_frey['soc_code_2010'].isin(old_codes)]
    aioe_codes = ';'.join(sorted(aioe_sources['SOC Code'].unique()))
    frey_codes = ';'.join(sorted(frey_sources['soc_code_2010'].unique()))
    check_value(soc, 'aioe', aioe_sources['AIOE'].mean(), 'AIOE_appendixA.xlsx + BLS crosswalk', aioe_codes)
    check_value(soc, 'fo_prob', frey_sources['fo_computerization_probability'].mean(), 'frey_osborne_probabilities.csv + BLS crosswalk', frey_codes)
    check_value(soc, 'aioe_source_codes', aioe_codes or pd.NA, 'AIOE_appendixA.xlsx + BLS crosswalk', aioe_codes)
    check_value(soc, 'fo_source_codes', frey_codes or pd.NA, 'frey_osborne_probabilities.csv + BLS crosswalk', frey_codes)
    check_value(soc, 'aioe_source_rows', len(aioe_sources) if len(aioe_sources) else pd.NA, 'AIOE_appendixA.xlsx', aioe_codes)
    check_value(soc, 'fo_source_rows', len(frey_sources) if len(frey_sources) else pd.NA, 'frey_osborne_probabilities.csv', frey_codes)

    title = mappings['2018 SOC Title'].str.replace(r'\s*\(#+\)', '', regex=True).str.strip().iloc[0]
    check_value(soc, 'title', title, 'soc_2010_to_2018_crosswalk.xlsx', soc)

    zones = raw_zones[raw_zones['soc2018'].eq(soc)]
    base_zone = zones[zones['O*NET-SOC Code'].eq(soc + '.00')]
    expected_zone = base_zone['Job Zone'].iloc[0] if len(base_zone) else zones['Job Zone'].mode().min()
    check_value(soc, 'job_zone', expected_zone, 'onet_job_zones.xlsx', ';'.join(zones['O*NET-SOC Code']))

    for feature in feature_dictionary.itertuples(index=False):
        source = raw_features[feature.source_workbook]
        values = source[
            source['soc2018'].eq(soc) & source['Element ID'].eq(feature.element_id) & source['Scale ID'].eq(feature.scale_id)
        ]
        check_value(soc, feature.column_name, values['Data Value'].mean(), feature.source_workbook,
                    ';'.join(values['O*NET-SOC Code']) + ' | ' + feature.element_id + ' | ' + feature.scale_id)

qa_trace = pd.DataFrame(qa_records)
qa_summary = qa_trace.groupby('soc2018').agg(checks=('passed', 'size'), passed=('passed', 'sum'))
qa_summary['failed'] = qa_summary['checks'] - qa_summary['passed']
qa_summary = qa_summary.join(master.set_index('soc2018')['title'])
qa_summary

,checks,passed,failed,title
soc2018,,,,
11-1011,197,197,0,Chief Executives
11-2032,197,197,0,Public Relations Managers
11-9179,197,197,0,"Personal Service Managers, All Other"
13-1082,197,197,0,Project Management Specialists
29-1299,197,197,0,Healthcare Diagnosing or Treating Practitioner...


In [17]:
# Inspect the source codes and expected/actual values for the main scores
qa_trace[qa_trace['column_name'].isin(['aioe', 'fo_prob', 'dv_rating_beta', 'human_rating_beta', 'job_zone', 'title'])]

,soc2018,column_name,source_file,source_rows,expected,actual,passed
1,11-1011,dv_rating_beta,occ_level.csv,11-1011.00;11-1011.03,0.507778,0.507778,True
4,11-1011,human_rating_beta,occ_level.csv,11-1011.00;11-1011.03,0.369444,0.369444,True
6,11-1011,aioe,AIOE_appendixA.xlsx + BLS crosswalk,11-1011,1.334246,1.334246,True
7,11-1011,fo_prob,frey_osborne_probabilities.csv + BLS crosswalk,11-1011,0.015,0.015,True
12,11-1011,title,soc_2010_to_2018_crosswalk.xlsx,11-1011,Chief Executives,Chief Executives,True
13,11-1011,job_zone,onet_job_zones.xlsx,11-1011.00;11-1011.03,5,5,True
198,11-9179,dv_rating_beta,occ_level.csv,11-9179.01;11-9179.02,0.452122,0.452122,True
201,11-9179,human_rating_beta,occ_level.csv,11-9179.01;11-9179.02,0.325311,0.325311,True
203,11-9179,aioe,AIOE_appendixA.xlsx + BLS crosswalk,11-9199,0.964687,0.964687,True
204,11-9179,fo_prob,frey_osborne_probabilities.csv + BLS crosswalk,11-9199,0.25,0.25,True


In [18]:
# Stop before exporting if any source trace failed
qa_failures = qa_trace[~qa_trace['passed']]
assert qa_failures.empty, qa_failures.to_string(index=False)
assert master['soc2018'].is_unique and master['soc2018'].notna().all()
assert len(master) == len(eloundou)
assert master['title'].notna().all()

print('All', len(qa_trace), 'source comparisons passed across five occupations.')
print('Final rows:', len(master), '| Unique jobs:', master['soc2018'].nunique())
print('Final columns:', len(master.columns), '| Dictionary entries:', len(data_dictionary))
print('Jobs outside the base (retained in coverage):', len(outside_base))
print('Unmatched original AIOE rows (kept separately):', len(aioe_unmatched))

All 985 source comparisons passed across five occupations.
Final rows: 798 | Unique jobs: 798
Final columns: 214 | Dictionary entries: 214
Jobs outside the base (retained in coverage): 10
Unmatched original AIOE rows (kept separately): 1


## 7. Export the two deliverables

Export only after the QA checks pass. CSV has no stored column types: reload `soc2018` as text. Empty fields mean missing information, not zero.

`merged_v0.csv` keeps all Eloundou jobs, including rows with missing scores or O*NET ratings. The notebook's `coverage`, `outside_base`, `aioe_unmatched`, and `join_log` retain the audit details. No earlier source or processed files are overwritten.

In [19]:
master.to_csv(processed_folder / 'merged_v0.csv', index=False)
data_dictionary.to_csv(processed_folder / 'data_dictionary.csv', index=False)

# Read back the deliverables to check their shape, keys, and dictionary coverage
saved = pd.read_csv(processed_folder / 'merged_v0.csv', dtype={'soc2018': 'string'})
saved_dictionary = pd.read_csv(processed_folder / 'data_dictionary.csv')

## Result and retained limitations

- One row per six-digit 2018 SOC job, anchored on the Task 2 Eloundou rollup.
- Sequential join counts and full outer coverage are shown above.
- Crosswalk scores are copied for splits and averaged when several old codes reach one new code. These unweighted averages are a documented v0 assumption.
- Official BLS titles resolve the 24 missing broad labels in the final table; the original Task 2 file and its title gaps remain unchanged.
- Five selected occupations are traced to source values before export.
- Missing scores and feature ratings remain missing. The 653-row Frey–Osborne source is still a subset of the paper's 702 occupations.
- The separate unmatched AIOE Biologists row is not assigned an invented code. Jobs outside the Eloundou base are visible in the coverage check and excluded from the left-joined deliverable.

This completes data assembly and the v0 QA check. It does not yet calculate ranks, agreement statistics, or model results.